## ByteAtlas: BTreeMap TLV Serialization
---

In [2]:
/*
Purpose: Apache-Avro is great, until you need zero copy with zeroizing on most allocations.
*/

In [3]:
:dep zeroize

In [4]:
use std::{collections::BTreeMap, borrow::Cow, io};
use zeroize::{Zeroize, Zeroizing};

In [5]:
struct ByteAtlas<'a> {
    entries: BTreeMap<u8, Cow<'a, [u8]>>,
}

In [6]:
impl<'a> ByteAtlas<'a> {
    pub fn deserialize(data: &'a [u8]) -> Self {
        let mut entries = BTreeMap::new();
        let mut cursor = 0;
        while cursor < data.len() {
            // T: Type (1 byte)
            let tag = data[cursor];
            cursor += 1;
            // L: Length (4 bytes, little endian)
            let len_bytes = &data[cursor..cursor + 4];
            let len = u32::from_le_bytes(len_bytes.try_into().unwrap()) as usize;
            cursor += 4;
            // V: Value (Borrows from slice of original)
            let value = &data[cursor..cursor + len];
            entries.insert(tag, Cow::Borrowed(value));
            cursor += len;
        }
        ByteAtlas { entries }
    }
    pub fn serialize(&self) -> Zeroizing<Vec<u8>> {
        let mut buffer = Vec::new();
        for (&tag, value) in &self.entries {
            // T: Type (1 byte)
            buffer.push(tag);
            // L: Length (4 bytes, little endian)
            let len = value.len() as u32;
            buffer.extend_from_slice(&len.to_le_bytes());
            // V: Value
            buffer.extend_from_slice(value.as_ref());
        }
        Zeroizing::new(buffer)
    }
    pub fn get(&self, tag: u8) -> Option<&[u8]> {
        self.entries.get(&tag).map(|cow| cow.as_ref())
    }
    pub fn get_str(&self, tag: u8) -> Option<&str> {
        self.get(tag).and_then(|bytes| std::str::from_utf8(bytes).ok())
    }
    pub fn set<V: Into<Cow<'a, [u8]>>>(&mut self, tag: u8, value: V) {
        self.entries.insert(tag, value.into());
    }
}

In [7]:
{
    let decrypted_file_buffer = vec![
        0x01, 0x05, 0x00, 0x00, 0x00, b'a', b'b', b'c', b'd', b'e',
        0x02, 0x03, 0x00, 0x00, 0x00, b'1', b'2', b'3',
    ];
    // Zero copy loading.
    let mut db = ByteAtlas::deserialize(&decrypted_file_buffer);
    // Modify data.
    db.set(0x02, b"topsecret");
    // Reserialize.
    let output = db.serialize();
    println!("{:?}", output);
    println!("{:?}", db.get(0x02));
    println!("{:?}", db.get_str(0x02));
};

Zeroizing([1, 5, 0, 0, 0, 97, 98, 99, 100, 101, 2, 9, 0, 0, 0, 116, 111, 112, 115, 101, 99, 114, 101, 116])
Some([116, 111, 112, 115, 101, 99, 114, 101, 116])
Some("topsecret")
